# Data Ingestion

In [204]:
# Using the pandas library, create a dataframe of the raw data file
import pandas as pd
import numpy as np

frailty_df = pd.read_csv('../Data_Raw/raw_frailty_data.csv')

# Data Processing

In [205]:
# Unit standardization
# Convert Height from inches to meters and Weight from pounds to kilograms
# Remane the columns changed to reflect the change in units
frailty_df['Height'] *= 0.0254
frailty_df['Weight'] *= 0.45359273
frailty_df.rename(columns={'Height': 'Height_m', 'Weight': 'Weight_kg'}, inplace=True)

# Feature Engineering
# Add a column for BMI = Weight_kg / Height_m ** 2
frailty_df['BMI'] = round(frailty_df['Weight_kg']/(frailty_df['Height_m']**2), 2)
# Add a column for AgeGroup (categorical). The cut() function sorts the ages into groups based on the bounds in bin
frailty_df['Age Group'] = pd.cut(frailty_df['Age'], bins=[0, 29, 45, 60, 150], labels=['<30', '30-45', '46-60', '>60'], right=True)

# Categorical to Numeric Encoding
# Convert frailty from Y or N to 0 or 1 (int8)
frailty_df['Frailty'] = frailty_df['Frailty'].map({'Y': 1, 'N': 0})
frailty_df['Frailty'] = frailty_df['Frailty'].astype('int8')
# One-hot encode Age Group into seperate columns
one_hot_encode = pd.get_dummies(frailty_df['Age Group'])
frailty_df.drop('Age Group', axis = 1, inplace=True)
frailty_df = frailty_df.join(one_hot_encode)

# Save the dataframe to the Data_Clean folder
frailty_df.to_csv('../Data_Clean/clean_frailty_data.csv')

# Data Analysis

In [206]:
# EDA & Reporting
# Mean, Median, std for numeric columns
summary = frailty_df.agg(['mean', 'median', 'std'])
# Write the findings to a markdown file
summary.to_markdown('../Reports/findings.md')

# Quantify Relation of Strength and Frailty
with open('../Reports/findings.md', 'a') as f:
    f.write('\n\n## Grip Strength to Frailty Correlation\n')
    f.write(str(frailty_df['Grip strength'].corr(frailty_df['Frailty'])))